# B3 local-refinement training -- SEED 2

Run this notebook and `B3_Multiseed_Seed3.ipynb` **at the same time, in two separate Colab tabs/runtimes**, so both finish in ~2h7m wall-clock instead of ~4h14m sequentially.

Trains one local-refinement run with `torch.manual_seed(2)`, identical in every other setting to the existing `b3_training_local_refine` run (same architecture, 50,000 iterations, `local_refine_n_sub=10`, reused `input_norm.json`), then runs both the regional-stress convergence sweep and the global QoI/Jacobian check on the result.

Once both this and the seed-3 notebook have finished, run the aggregation cell at the bottom of **either** notebook (or `B3_Multiseed_Aggregate.ipynb`) to get mean ± std across all 3 seeds.

In [ ]:
import os, subprocess, sys, time

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'timm', 'h5py', 'jax', 'tqdm'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)
for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

R = '/content/drive/MyDrive/pfem_run'
RUN4_DIR = f'{R}/b3_training_normalized'
print('Ready.')

In [ ]:
SEED = 2
import shutil
from omar_pfem.train_B3 import get_args, train, DEFAULT_RESOLUTION

run_dir = f'{R}/b3_training_local_refine_seed{SEED}'
os.makedirs(run_dir, exist_ok=True)

norm_src = f'{RUN4_DIR}/input_norm.json'
norm_dst = f'{run_dir}/input_norm.json'
assert os.path.exists(norm_src), f'baseline input_norm.json not found: {norm_src}'
shutil.copy(norm_src, norm_dst)

args = get_args([
    '--n_iters', '50000',
    '--batch_size', '8',
    '--log_every', '500',
    '--ckpt_every', '50000',
    '--output_dir', run_dir,
    '--normalize_inputs', '1',
    '--local_refine_region', '1',
    '--local_refine_n_sub', '10',
])

print(f'SEED {SEED}: training (50,000 iters, local_refine_n_sub=10, same config as the '
      f'original run, only the weight-init seed differs)')
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.reset_peak_memory_stats(device)
t0 = time.time()
model = train(args, device)
elapsed = time.time() - t0
print(f'SEED {SEED}: training finished in {elapsed:.1f}s ({elapsed/args.n_iters:.4f}s/iter).')

In [ ]:
checkpoint = f'{run_dir}/checkpoint_{args.n_iters}.pt'
dataset = f'{R}/b3_dataset_clean_holdout/dataset.h5'

print(f'SEED {SEED}: regional-stress convergence sweep...')
sys.argv = [
    'evaluate_B3_region_local_refine.py',
    '--checkpoint', checkpoint, '--dataset', dataset,
    '--out_json', f'{run_dir}/region_local_refine_convergence.json',
    '--n_sub_sweep', '2', '4', '6', '8', '10', '12',
]
from omar_pfem.evaluate_B3_region_local_refine import main as eval_region_main
eval_region_main()

print(f'\nSEED {SEED}: global QoIs + Jacobian check...')
sys.argv = [
    'evaluate_B3_global_qois_and_jacobian.py',
    '--checkpoint', checkpoint, '--dataset', dataset,
    '--out_json', f'{run_dir}/global_qois_and_jacobian.json',
]
from omar_pfem.evaluate_B3_global_qois_and_jacobian import main as eval_global_main
eval_global_main()

print(f'\nSEED {SEED} DONE. Results saved under {run_dir}.')